In [13]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from pathlib import Path

# Imports du projet
import sys
sys.path.append(str(Path.cwd().parent))
from mpp_project.core import load_tournament_data, generate_drifted_ensemble_crowds
from mpp_project.v5_supervised.oracle_dp import backpropagate_group_stages_stochastic

# --- CONFIGURATION INITIALE ---
N_OPPONENTS = 11
GRID_SIZE = 1001
GAP_OFFSET = 600
HAS_BOOSTER = 1 

# 1. Chargement des données
notebook_dir = Path.cwd()
project_root = notebook_dir.parent 
data_path_csv = project_root / "data" / "CDM_2026_imputed.csv"
data_path_market = project_root / "data" / "CDM_2026_goal_scorer_and_favorite.csv"
data_path_npy = project_root / "data" / "expected_V_phases_finales.npy"

# Chargement de la matrice d'horizon des phases finales
V_all_matches = np.load(data_path_npy)
# L'index 0 du .npy correspond à la fin des poules
V_start_phases_finales = V_all_matches[0] 

# --- CORRECTION : TRAVERSÉE DES POULES ---
print("--- BACKPROPAGATION DES POULES (Création de V_match_0) ---")
# On charge tous les matchs
df_matches, true_probas_all, mpp_gains_all, crowd_repartitions_all = load_tournament_data(data_path_csv)

# Filtrage pour les poules
is_poule = df_matches['phase'].str.contains('Poule', na=False)
n_poules = is_poule.sum()

probas_poules = true_probas_all[:n_poules].astype(np.float32)
gains_poules = mpp_gains_all[:n_poules].astype(np.int32)
crowds_poules = crowd_repartitions_all[:n_poules].astype(np.float32)

# On est au tout début de la compétition
MATCH_ACTUEL = 0 
N_ENSEMBLES = 15 

# 1. Génération du tenseur des foules futures
print(f"Génération de {N_ENSEMBLES} univers de foules avec dégradation temporelle...")
ensemble_crowds = generate_drifted_ensemble_crowds(
    MATCH_ACTUEL, 
    probas_poules, 
    crowds_poules, 
    n_ensembles=N_ENSEMBLES
)

# 2. Backpropagation C++
print("Rétropropagation de la matrice 4D...")
V_match_0 = backpropagate_group_stages_stochastic(
    MATCH_ACTUEL,
    probas_poules, 
    gains_poules, 
    ensemble_crowds, 
    V_start_phases_finales
)
print("✅ Matrice V_match_0 calculée avec succès ! Prêt pour les Buteurs/Favoris.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
--- BACKPROPAGATION DES POULES (Création de V_match_0) ---
Génération de 15 univers de foules avec dégradation temporelle...
Rétropropagation de la matrice 4D...
✅ Matrice V_match_0 calculée avec succès ! Prêt pour les Buteurs/Favoris.


In [17]:
# --- 2. TRAITEMENT DES DONNÉES ET PROBABILITÉS ---
df_market = pd.read_csv(data_path_market)

# Séparation
df_fav = df_market[df_market['category'] == 'favorite'].copy()
df_sco = df_market[df_market['category'] == 'scorer'].copy()

# A. Traitement des Favoris (Classique)
raw_implied_fav = 1.0 / df_fav['cote'].values
df_fav['true_proba'] = raw_implied_fav / raw_implied_fav.sum()

# B. Traitement des Buteurs (Logique "Autre")
# On isole la ligne "Autre" (qui n'a pas de cote)
autre_mask = df_sco['selection'].str.lower() == 'autre'
df_sco_autre = df_sco[autre_mask].copy()
df_sco_individuals = df_sco[~autre_mask].copy()

# Calcul des vraies probas sur tous les buteurs individuels (MPP + Non-MPP)
raw_implied_sco = 1.0 / df_sco_individuals['cote'].values
df_sco_individuals['true_proba'] = raw_implied_sco / raw_implied_sco.sum()

# Séparation des buteurs jouables sur MPP (gain_mpp renseigné) et les "Fantômes" (gain_mpp vide/NaN)
mpp_scorers = df_sco_individuals[df_sco_individuals['gain_mpp'].notna()].copy()
ghost_scorers = df_sco_individuals[df_sco_individuals['gain_mpp'].isna()].copy()

# La probabilité de "Autre" est la somme des probabilités des fantômes
proba_autre = ghost_scorers['true_proba'].sum()

# On reconstruit la liste finale jouable sur MPP
if not df_sco_autre.empty:
    print(f"✅ 'Autre' trouvé dans le CSV, sa probabilité est {proba_autre:.4f}")
    df_sco_autre['true_proba'] = proba_autre
    # On force la cote à NaN car elle ne sert plus, et on s'assure que le gain est bien lu
    df_mpp_scorers = pd.concat([mpp_scorers, df_sco_autre], ignore_index=True)
else:
    print(f"⚠️ 'Autre' non trouvé dans le CSV, il sera créé avec une probabilité de {proba_autre:.4f}")
    # Au cas où "Autre" n'est pas dans le CSV, on le crée
    df_mpp_scorers = mpp_scorers.copy()

✅ 'Autre' trouvé dans le CSV, sa probabilité est 0.3357


In [18]:
# --- 3. MODÉLISATION DU CROWD MPP (Le Biais Chauvin / Héroïque) ---
def estimate_mpp_crowd(selections, true_probs, gains, base_beta=1.2):
    """
    Remplace le crowd Winamax par un modèle comportemental adapté à MPP.
    """
    crowd_weights = true_probs ** base_beta
    
    # Biais de désirabilité / Chauvinisme
    chauvinism_multiplier = np.ones(len(selections))
    
    for i, sel in enumerate(selections):
        sel_lower = str(sel).lower()
        
        # Biais Énorme : La France et ses joueurs
        if sel_lower in ['france', 'kylian_mbappe']:
            chauvinism_multiplier[i] = 6.5
            
        # Biais Fort : Les superstars internationales (Ronaldo, Messi si présents)
        elif sel_lower in ['portugal', 'cristiano_ronaldo', 'messi', 'argentine', 'bresil']:
            chauvinism_multiplier[i] = 1.5
            
    # La catégorie "Autre" est historiquement sous-jouée par les casuals (biais de représentativité)
    for i, sel in enumerate(selections):
        if str(sel).lower() == 'autre':
            chauvinism_multiplier[i] = 0.9
            
    final_weights = crowd_weights * chauvinism_multiplier
    return final_weights / final_weights.sum()

# Application du modèle (on écrase la colonne crowd du CSV)
df_fav['estimated_crowd'] = estimate_mpp_crowd(df_fav['selection'].values, df_fav['true_proba'].values, df_fav['gain_mpp'].values)
df_mpp_scorers['estimated_crowd'] = estimate_mpp_crowd(df_mpp_scorers['selection'].values, df_mpp_scorers['true_proba'].values, df_mpp_scorers['gain_mpp'].values)

print("\n" + "="*50)
print("🔍 VÉRIFICATION DES PARAMÈTRES : FAVORIS")
print("="*50)
display_fav = df_fav[['selection', 'cote', 'gain_mpp', 'true_proba', 'estimated_crowd']].copy()
# Formatage pour une lecture propre
display_fav['true_proba'] = (display_fav['true_proba'] * 100).map('{:.1f}%'.format)
display_fav['estimated_crowd'] = (display_fav['estimated_crowd'] * 100).map('{:.1f}%'.format)
print(display_fav.sort_values(by='estimated_crowd', ascending=False).to_string(index=False))

print("\n" + "="*50)
print("🔍 VÉRIFICATION DES PARAMÈTRES : BUTEURS")
print("="*50)
display_sco = df_mpp_scorers[['selection', 'cote', 'gain_mpp', 'true_proba', 'estimated_crowd']].copy()
display_sco['true_proba'] = (display_sco['true_proba'] * 100).map('{:.1f}%'.format)
display_sco['estimated_crowd'] = (display_sco['estimated_crowd'] * 100).map('{:.1f}%'.format)
print(display_sco.sort_values(by='estimated_crowd', ascending=False).to_string(index=False))


🔍 VÉRIFICATION DES PARAMÈTRES : FAVORIS
 selection  cote  gain_mpp true_proba estimated_crowd
   espagne   5.5      80.0      18.6%            9.5%
    bresil   9.0     130.0      11.3%            7.9%
 argentine  10.0     120.0      10.2%            7.0%
  portugal  11.0     110.0       9.3%            6.2%
angleterre   8.0     100.0      12.8%            6.1%
    france   6.0      90.0      17.0%           55.8%
 allemagne  17.0     150.0       6.0%            2.5%
  pays_bas  30.0     200.0       3.4%            1.2%
   norvege  40.0     230.0       2.6%            0.9%
  belgique  50.0     250.0       2.0%            0.7%
  colombie  50.0     400.0       2.0%            0.7%
     japon  60.0     400.0       1.7%            0.5%
     maroc  60.0     400.0       1.7%            0.5%
etats_unis  75.0     300.0       1.4%            0.4%

🔍 VÉRIFICATION DES PARAMÈTRES : BUTEURS
        selection  cote  gain_mpp true_proba estimated_crowd
       harry_kane   7.5     100.0      12.7%   

In [19]:
# --- 4. LE MOTEUR DE PROJECTION "MATCH 0" ---
print("--- SIMULATION STRATÉGIQUE DU MATCH 0 ---")

fav_selections = df_fav['selection'].tolist()
fav_gains = df_fav['gain_mpp'].values
fav_probs = df_fav['true_proba'].values
fav_crowd = df_fav['estimated_crowd'].values

sco_selections = df_mpp_scorers['selection'].tolist()
sco_gains = df_mpp_scorers['gain_mpp'].values
sco_probs = df_mpp_scorers['true_proba'].values
sco_crowd = df_mpp_scorers['estimated_crowd'].values

matrix_results = []

for idx_my_fav, my_fav_name in enumerate(fav_selections):
    for idx_my_sco, my_sco_name in enumerate(sco_selections):
        
        expected_win_rate = 0.0
        expected_pure_points = 0.0
        
        for f_star in range(len(fav_selections)):
            p_f = fav_probs[f_star]
            my_fav_pts = fav_gains[idx_my_fav] if idx_my_fav == f_star else 0
            
            for s_star in range(len(sco_selections)):
                p_s = sco_probs[s_star]
                my_sco_pts = sco_gains[idx_my_sco] if idx_my_sco == s_star else 0
                
                p_universe = p_f * p_s
                my_total_bonus = my_fav_pts + my_sco_pts
                expected_pure_points += p_universe * my_total_bonus
                
                # Le peloton est modélisé par son espérance pour gagner en rapidité (loi des grands nombres)
                pack_fav_pts = fav_crowd[f_star] * fav_gains[f_star]
                pack_sco_pts = sco_crowd[s_star] * sco_gains[s_star]
                pack_total_bonus = pack_fav_pts + pack_sco_pts
                
                # Le gap projeté contre le joueur moyen du peloton
                gap_pack = my_total_bonus - pack_total_bonus
                
                g1_next = max(-600, min(400, gap_pack))
                g2_next = max(-600, min(400, gap_pack)) 
                
                # LA VRAIE PROJECTION A LIEU ICI :
                # On lit le Win Rate dans V_match_0, pas dans V_phases_finales !
                wr_scenario = V_match_0[int(g1_next) + GAP_OFFSET, int(g2_next) + GAP_OFFSET, HAS_BOOSTER]
                
                expected_win_rate += p_universe * wr_scenario
                        
        matrix_results.append({
            'Favori': my_fav_name,
            'Buteur': my_sco_name,
            'EV Points': np.round(expected_pure_points, 1),
            'WR (Projection)': expected_win_rate
        })

# --- 5. RÉSULTATS ---
df_res = pd.DataFrame(matrix_results)
df_res = df_res.sort_values(by='WR (Projection)', ascending=False).reset_index(drop=True)

print("\n🏆 TOP 10 DES COMBINAISONS OPTIMALES :")
print(df_res.head(10).to_string(formatters={'WR (Projection)': '{:,.2%}'.format}))

print("\n📉 TOP 5 DES PIÈGES (BONS POINTS MAIS MAUVAIS WIN RATE) :")
# Tri par différence entre le classement EV et le classement WR
df_res['EV_Rank'] = df_res['EV Points'].rank(ascending=False)
df_res['WR_Rank'] = df_res['WR (Projection)'].rank(ascending=False)
df_res['Trap_Score'] = df_res['WR_Rank'] - df_res['EV_Rank'] 
traps = df_res.sort_values(by='Trap_Score', ascending=False).head(5)
print(traps[['Favori', 'Buteur', 'EV Points', 'WR (Projection)']].to_string(formatters={'WR (Projection)': '{:,.2%}'.format}))

--- SIMULATION STRATÉGIQUE DU MATCH 0 ---

🏆 TOP 10 DES COMBINAISONS OPTIMALES :
       Favori Buteur  EV Points WR (Projection)
0      bresil  autre       65.1          41.67%
1     espagne  autre       65.2          41.64%
2      france  autre       65.7          41.59%
3  angleterre  autre       63.1          41.55%
4   argentine  autre       62.6          41.54%
5    portugal  autre       60.6          41.42%
6   allemagne  autre       59.4          41.39%
7    colombie  autre       58.5          41.34%
8    pays_bas  autre       57.2          41.28%
9       maroc  autre       57.2          41.27%

📉 TOP 5 DES PIÈGES (BONS POINTS MAIS MAUVAIS WIN RATE) :
         Favori           Buteur  EV Points WR (Projection)
248  etats_unis  viktor_gyokeres       16.7          39.12%
186    pays_bas  viktor_gyokeres       19.5          39.24%
207     norvege  viktor_gyokeres       18.5          39.20%
221    belgique  viktor_gyokeres       17.8          39.17%
166     norvege        luis_diaz 